# Controlled Generation Pilot

In [1]:
import json
import time
from collections import defaultdict
from pathlib import Path

from edu_eval.generation.cases import load_cases, select_pilot, validation_report
from edu_eval.generation.config import GenerationConfig
from edu_eval.generation.controlled import ControlledGenerationRunner
from edu_eval.generation.runner import GenerationRunner
from edu_eval.models.audit import inspect_model
from edu_eval.models.loader import ModelLoader
from edu_eval.models.registry import ModelRegistry

In [2]:
PROJECT_ROOT = Path.cwd().parent
MODELS_CONFIG = PROJECT_ROOT / "configs" / "models.yaml"
GENERATION_CONFIG = PROJECT_ROOT / "configs" / "generation.yaml"
CASE_PATH = PROJECT_ROOT / "data" / "processed" / "controlled_generation_cases.jsonl"
RESULTS_DIR = PROJECT_ROOT / "results" / "generation" / "pilot"
model_id = "smoke_qwen"
generation_config = GenerationConfig.from_yaml(GENERATION_CONFIG)
print(model_id, generation_config.prompt_version, list(generation_config.target_grades))

smoke_qwen cg_v1 ['SD6', 'SMP7', 'SMP9', 'SMA10']


In [3]:
cases = load_cases(CASE_PATH)
report = validation_report(cases, generation_config)
checks = {k: report[k] for k in ("schema", "provenance", "curriculum_evidence", "phase_mapping", "subjects_match", "duplicate_case_ids", "pilot_ready", "full_ready", "research_ready")}
print(f"cases={report['total_cases']} {checks}")

cases=60 {'schema': 'PASS', 'provenance': 'DRAFT', 'curriculum_evidence': 'PASS', 'phase_mapping': 'PASS', 'subjects_match': 'PASS', 'duplicate_case_ids': 'PASS', 'pilot_ready': 'PASS', 'full_ready': 'PASS', 'research_ready': "NOT_READY: ['60 project_capsule references', '60 unknown training exposure', '240 phase_only evidence entries']"}


In [4]:
pilot_cases = select_pilot(cases, generation_config)
print(f"pilot={len(pilot_cases)}")
for case in pilot_cases:
    print(f"{case.case_id} {case.subject} {case.concept}")

pilot=10
CG-013 Bahasa Indonesia Ide pokok paragraf
CG-014 Bahasa Indonesia Kalimat efektif
CG-001 IPA Fotosintesis
CG-002 IPA Rantai makanan dan ekosistem
CG-049 IPS Kenampakan alam dan peta
CG-050 IPS Sumber daya alam
CG-037 Informatika Data dan informasi
CG-038 Informatika Algoritma
CG-025 Pendidikan Pancasila Sila Ketuhanan Yang Maha Esa
CG-026 Pendidikan Pancasila Sila Kemanusiaan yang Adil dan Beradab


In [5]:
registry = ModelRegistry.from_yaml(MODELS_CONFIG)
spec = registry.get(model_id)
tokenizer, model, device = ModelLoader.load(spec)
info = inspect_model(model, tokenizer)
print(f"{info.architecture} {info.parameter_count} {info.dtype} {info.device}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForCausalLM 494032768 bfloat16 cuda:0


In [6]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
model_out_dir = RESULTS_DIR / model_id
model_out_dir.mkdir(parents=True, exist_ok=True)
output_path = model_out_dir / "controlled_outputs.jsonl"
done: set[tuple[str, str]] = set()
if output_path.exists():
    with output_path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                r = json.loads(line)
                done.add((r["case_id"], r["target_grade"]))
generation_runner = GenerationRunner(model=model, tokenizer=tokenizer, device=device, config=generation_config)
controlled_runner = ControlledGenerationRunner(generation_runner=generation_runner, model_id=model_id, target_grades=list(generation_config.target_grades), prompt_version=generation_config.prompt_version)
total_expected = len(pilot_cases) * len(generation_config.target_grades)
t0 = time.perf_counter()
with output_path.open("a", encoding="utf-8") as f:
    for case in pilot_cases:
        for grade in generation_config.target_grades:
            if (case.case_id, grade) in done:
                continue
            result = controlled_runner.generate(case, grade)
            f.write(json.dumps(result, ensure_ascii=False) + "\n")
            f.flush()
            done.add((result["case_id"], result["target_grade"]))
print(f"done {len(done)}/{total_expected} {(time.perf_counter() - t0) / 60:.1f} min")

done 40/40 0.0 min


In [7]:
with output_path.open("r", encoding="utf-8") as f:
    outputs = [json.loads(line) for line in f if line.strip()]
by_grade: dict[str, list[dict]] = defaultdict(list)
for r in outputs:
    by_grade[r["target_grade"]].append(r)
for grade in generation_config.target_grades:
    rs = by_grade[grade]
    avg_out = sum(r["output_tokens"] for r in rs) / len(rs)
    trunc = sum(r["hit_max_new_tokens"] for r in rs)
    print(f"{grade}: n={len(rs)} avg_out={avg_out:.1f} trunc={trunc}")

SD6: n=10 avg_out=255.1 trunc=9
SMP7: n=10 avg_out=253.8 trunc=9
SMP9: n=10 avg_out=256.0 trunc=10
SMA10: n=10 avg_out=256.0 trunc=10


In [8]:
sample = pilot_cases[0]
print(f"{sample.case_id} {sample.subject} {sample.concept}")
for grade in generation_config.target_grades:
    r = next(x for x in outputs if x["case_id"] == sample.case_id and x["target_grade"] == grade)
    print(f"[{grade}] {r['output_text'][:200]}")

CG-013 Bahasa Indonesia Ide pokok paragraf
[SD6] Materi Pembelajaran: Ide Pokok Paragraf

Sebagai seorang siswa di sekolah dasar, kita harus memahami bagaimana menginterpretasikan dan menangkap ide pokok dari setiap paragraf. Ini adalah langkah awal
[SMP7] Materi Pembelajaran Bahasa Indonesia - Ide Pokok Paragraf

Sebagai seorang siswa SMA kelas 7, Anda telah menemukan materi ini sangat penting untuk memahami konsep ide pokok paragraf. Ia adalah bagian 
[SMP9] Materi Pembelajaran Bahasa Indonesia - Ide Pokok Paragraf

Sebagai pendidik SMP kelas 9, kita akan belajar tentang bagaimana menginterpretasi dan memahami ide pokok dari setiap paragraf. Ini adalah la
[SMA10] Materi Pembelajaran Bahasa Indonesia - Ide Pokok Paragraf

Sebagai seorang siswa SMA kelas 10, Anda telah menemukan materi ini sangat penting untuk memahami konsep ide pokok paragraf. Ia adalah bagian


In [9]:
from edu_eval.linguistic.mentions import classify_mentions

flags = [classify_mentions(r["output_text"]) for r in outputs]
for key in ("grade_mention", "instruction_disclosure", "meta_prompt_disclosure"):
    print(f"{key}={sum(f[key] for f in flags)}/{len(flags)}")

grade_mention=27/40
instruction_disclosure=6/40
meta_prompt_disclosure=0/40


Acceptance checklist: README.md